# 物流时效

只看已送达订单，其余状态没有送达时间。`slack_days` 为正表示比承诺早到。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
query("""
SELECT COUNT(*) AS delivered_orders,
       ROUND(AVG(delivered_days), 2) AS avg_delivered_days,
       ROUND(AVG(promise_days), 2) AS avg_promise_days,
       ROUND(AVG(slack_days), 2) AS avg_slack_days,
       ROUND(AVG(slack_days >= 0) * 100, 2) AS on_time_pct
FROM (
    SELECT order_id,
           MAX(delivered_days) AS delivered_days,
           MAX(promise_days) AS promise_days,
           MAX(slack_days) AS slack_days
    FROM dws_order_wide
    WHERE order_status = 'delivered'
    GROUP BY order_id
) t
""")


,delivered_orders,avg_delivered_days,avg_promise_days,avg_slack_days,on_time_pct
0,96478,12.56,23.74,11.18,91.89


## 按州看准时率

In [3]:
query("""
SELECT CASE
         WHEN slack_days >= 7 THEN '提前7天以上'
         WHEN slack_days >= 0 THEN '提前0-7天'
         WHEN slack_days > -7 THEN '延迟1-7天'
         ELSE '延迟7天以上'
       END AS bucket,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_score,
       ROUND(AVG(review_score <= 2) * 100, 2) AS bad_review_pct
FROM (
    SELECT order_id,
           MAX(slack_days) AS slack_days,
           MAX(review_score) AS review_score
    FROM dws_order_wide
    WHERE order_status = 'delivered' AND review_score IS NOT NULL
    GROUP BY order_id
) t
GROUP BY bucket
ORDER BY avg_score DESC
""")


,bucket,orders,avg_score,bad_review_pct
0,提前7天以上,70929,4.32,8.94
1,提前0-7天,17234,4.20,10.21
2,延迟1-7天,4409,3.18,36.04
3,延迟7天以上,3260,1.74,78.16


## 按距离分档

In [4]:
query("""
SELECT CASE
         WHEN distance_km <= 100 THEN '100km内'
         WHEN distance_km <= 500 THEN '100-500km'
         WHEN distance_km <= 1000 THEN '500-1000km'
         ELSE '1000km以上'
       END AS dist_bucket,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(AVG(slack_days < 0) * 100, 2) AS late_pct
FROM dws_order_wide
WHERE order_status = 'delivered' AND distance_km IS NOT NULL
GROUP BY dist_bucket
ORDER BY orders DESC
""")

,dist_bucket,orders,late_pct
0,100-500km,37187,6.85
1,500-1000km,25897,8.24
2,100km内,17998,6.39
3,1000km以上,15401,11.69


## 延误与评分

把订单按「提前/延迟」分档，看平均评分和差评率怎么变。

In [5]:
query("""
SELECT CASE
         WHEN slack_days >= 7 THEN '提前7天以上'
         WHEN slack_days >= 0 THEN '提前0-7天'
         WHEN slack_days > -7 THEN '延迟1-7天'
         ELSE '延迟7天以上'
       END AS bucket,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_score,
       ROUND(AVG(review_score <= 2) * 100, 2) AS bad_review_pct
FROM dws_order_wide
WHERE order_status = 'delivered' AND review_score IS NOT NULL
GROUP BY bucket
ORDER BY avg_score DESC
""")

,bucket,orders,avg_score,bad_review_pct
0,提前7天以上,81378,4.23,11.25
1,提前0-7天,19465,4.14,11.85
2,延迟1-7天,4932,3.15,37.19
3,延迟7天以上,3595,1.73,78.39
